# Restrições e Heurísticas do Projeto CSP

Este notebook define as hard constraints e funções de avaliação/otimização usadas no projeto de horários (CSP).

Resumo das restrições implementadas:
- `AllDifferentAttrConstraint(attr)`: garante que, num conjunto de variáveis, o atributo `attr` (por exemplo, `bloco`) não se repete.
- `not_same_room(aula1, aula2)`: impede duas aulas na mesma sala e no mesmo bloco (ignora aulas "Online").
- `MaxAulasPorDiaConstraint(max_por_dia=3)`: limita o número de aulas por dia para um mesmo conjunto (ex.: por turma).
- `OnlineMax3SameDayConstraint`: quando houver aulas online para uma turma, no máximo 3 e todas no mesmo dia.

Também são definidas funções de penalização (para soft constraints) e a heurística de busca local `hill_climbing`.


In [1]:
from constraint import AllDifferentConstraint, Constraint
from graph import bloco_para_dia
import random
from collections import defaultdict, Counter
import copy
import math


## AllDifferentAttrConstraint e not_same_room

- `AllDifferentAttrConstraint(attr)`: extensão de `AllDifferentConstraint` que compara um atributo de objetos atribuídos (por exemplo, `aula.bloco`).
- `not_same_room(aula1, aula2)`: duas aulas não podem ocorrer na mesma sala no mesmo bloco; aulas "Online" não conflitam em sala.


In [2]:
class AllDifferentAttrConstraint(AllDifferentConstraint):
    def __init__(self, attr):
        super().__init__()
        self.attr = attr

    def __call__(self, variables, domains, assignments, forwardcheck=False):
        seen = set()
        for var in variables:
            if var in assignments:
                val = getattr(assignments[var], self.attr)
                if val in seen:
                    return False
                seen.add(val)
        return True

def not_same_room(aula1, aula2):
    if aula1.sala == "Online" or aula2.sala == "Online":
        return True
    return not (aula1.bloco == aula2.bloco and aula1.sala == aula2.sala)


## MaxAulasPorDiaConstraint

Limita o número de aulas por dia no conjunto de variáveis alvo (ex.: todas as variáveis de uma turma), com um máximo definido por `max_por_dia`.


In [3]:
class MaxAulasPorDiaConstraint(Constraint):
    def __init__(self, max_por_dia=3):
        self.max_por_dia = max_por_dia

    def __call__(self, variables, domains, assignments, forwardcheck=False):
        contagem_dias = defaultdict(int)
        for var in variables:
            if var in assignments:
                aula = assignments[var]
                dia = bloco_para_dia(aula.bloco)
                contagem_dias[dia] += 1
                if contagem_dias[dia] > self.max_por_dia:
                    return False
        return True


## OnlineMax3SameDayConstraint

Restrição rígida para aulas online por turma:
- Se houver aulas online, o total deve ser no máximo 3.
- Se houver 1 a 3 aulas online, todas devem ocorrer no mesmo dia.


In [ ]:
class OnlineMax3SameDayConstraint(Constraint):
    def __call__(self, variables, domains, assignments, forwardcheck=False):
        aulas_por_turma = defaultdict(list)

        for var in variables:
            if var in assignments and getattr(assignments[var], "sala", None) == "Online":
                partes = var.split("_") 
                turma = partes[3]
                aulas_por_turma[turma].append(assignments[var])

        for aulas_online in aulas_por_turma.values():
            total = len(aulas_online)
            if total > 3:
                return False
            if 1 <= total <= 3:
                dias = {bloco_para_dia(a.bloco) for a in aulas_online}
                if len(dias) != 1:
                    return False

        return True


## Penalizações (soft constraints)

As funções abaixo calculam penalizações para a solução, usadas pela heurística de melhoria:
- `penalidade_uc_dias_distintos`: penaliza quando as aulas da mesma UC, para a mesma turma, caem repetidas no mesmo dia.
- `penalidade_max_4_dias_por_turma`: penaliza turmas com carga distribuída em 4 ou mais dias.
- `penalidade_aulas_consecutivas`: penaliza ausências de consecutividade intra-dia (ajusta a preferência de blocos contíguos).
- `penalidade_aulas_sozinhas`: penaliza dias em que uma turma tem apenas uma aula.
- `penalidade_min_salas_por_turma_por_dia`: penaliza uso de múltiplas salas no mesmo dia para a mesma turma.
- `pontuacao`: soma ponderada (aqui simples soma) das penalizações.


In [6]:
def penalidade_uc_dias_distintos(solucao):
    penalidade = 0
    ucs_por_turma = defaultdict(list)
    for var, aula in solucao.items():
        if var.startswith("aula_"):
            _, uc, _, turma, _ = var.split("_")
            ucs_por_turma[(turma, uc)].append(aula)

    for aulas_uc in ucs_por_turma.values():
        dias = [bloco_para_dia(aula.bloco) for aula in aulas_uc]
        contador = Counter(dias)
        for _, quantidade in contador.items():
            if quantidade > 1:
                penalidade += 2 * (quantidade - 1)
    return penalidade

def penalidade_max_4_dias_por_turma(solucao):
    penalidade = 0
    turmas = defaultdict(set)
    for var, aula in solucao.items():
        if var.startswith("aula_"):
            partes = var.split("_")
            _, _, _, turma, _ = partes
            dia = bloco_para_dia(aula.bloco)
            turmas[turma].add(dia)
    for dias_turma in turmas.values():
        if len(dias_turma) >= 4:
            penalidade += (len(dias_turma) - 4)*2
    return penalidade

def penalidade_aulas_consecutivas(solucao):
    penalidade = 0
    turmas = defaultdict(lambda: defaultdict(list))
    for var, aula in solucao.items():
        if var.startswith("aula_"):
            _, _, _, turma, _ = var.split("_")
            dia = bloco_para_dia(aula.bloco)
            turmas[turma][dia].append(aula.bloco)
    for _, dias in turmas.items():
        for blocos in dias.values():
            blocos.sort()
            for i in range(1, len(blocos)):
                if blocos[i] != blocos[i-1] + 1:
                    penalidade += 1
    return penalidade

def penalidade_aulas_sozinhas(solucao):

    penalidade = 0
    turmas = defaultdict(lambda: defaultdict(list))  

    for var, aula in solucao.items():
        if var.startswith("aula_"):
            partes = var.split("_")
            _, _, _, turma, _ = partes
            dia = bloco_para_dia(aula.bloco)
            turmas[turma][dia].append(aula.bloco)

    for turma, dias in turmas.items():
        for blocos in dias.values():
            if len(blocos) == 1:
                penalidade += 2  

    return penalidade

def penalidade_min_salas_por_turma_por_dia(solucao):
    penalidade = 0
    turmas_por_dia = defaultdict(lambda: defaultdict(set))
    for var, aula in solucao.items():
        if var.startswith("aula_"):
            turma = var.split("_")[3]
            dia = bloco_para_dia(aula.bloco)
            turmas_por_dia[turma][dia].add(aula.sala)
    for _, dias in turmas_por_dia.items():
        for salas in dias.values():
            if len(salas) > 1:
                penalidade += len(salas) - 1
    return penalidade

def pontuacao(solucao):
    return (penalidade_uc_dias_distintos(solucao) +
            penalidade_max_4_dias_por_turma(solucao) +
            penalidade_aulas_consecutivas(solucao) +
            penalidade_aulas_sozinhas(solucao) +
            penalidade_min_salas_por_turma_por_dia(solucao))


## Utilitários de verificação de hard constraints durante a heurística

A função `violates_hard_constraints_for_move` impede movimentos que quebrem hard constraints quando o `hill_climbing` tenta ajustar blocos.


In [ ]:
MAX_AULAS_POR_DIA = 3

def get_parts(var_name):
    partes = var_name.split("_") 
    return partes[1], partes[2], partes[3]

def violates_hard_constraints_for_move(solucao, var, bloco_novo, all_variables):
    uc, professor, turma = get_parts(var)
    nova_aula = copy.deepcopy(solucao[var])
    nova_aula.bloco = bloco_novo

    for other_var, other_aula in solucao.items():
        if other_var == var or not other_var.startswith("aula_"):
            continue
        _, _, other_turma = get_parts(other_var)
        if other_turma == turma and other_aula.bloco == bloco_novo:
            return True

    for other_var, other_aula in solucao.items():
        if other_var == var or not other_var.startswith("aula_"):
            continue
        _, other_prof, _ = get_parts(other_var)
        if other_prof == professor and other_aula.bloco == bloco_novo:
            return True

    sala_nova = getattr(nova_aula, "sala", None)
    if sala_nova != "Online":
        for other_var, other_aula in solucao.items():
            if other_var == var or not other_var.startswith("aula_"):
                continue
            if getattr(other_aula, "sala", None) == sala_nova and other_aula.bloco == bloco_novo:
                return True

    contagem_dias = defaultdict(int)
    for other_var, other_aula in solucao.items():
        if not other_var.startswith("aula_"):
            continue
        _, _, other_turma = get_parts(other_var)
        if other_turma != turma or other_var == var:
            continue
        contagem_dias[bloco_para_dia(other_aula.bloco)] += 1

    contagem_dias[bloco_para_dia(bloco_novo)] += 1
    if any(cont > MAX_AULAS_POR_DIA for cont in contagem_dias.values()):
        return True

    online_aulas = []
    for other_var, other_aula in solucao.items():
        if not other_var.startswith("aula_"):
            continue
        _, _, other_turma = get_parts(other_var)
        if other_turma != turma or other_var == var:
            continue
        if getattr(other_aula, "sala", None) == "Online":
            online_aulas.append(other_aula)

    if sala_nova == "Online":
        online_aulas.append(nova_aula)

    total_online = len(online_aulas)
    if total_online > 3:
        return True
    if 1 <= total_online <= 3:
        dias_online = {bloco_para_dia(a.bloco) for a in online_aulas}
        if len(dias_online) != 1:
            return True

    return False


## Heurística de melhoria: hill_climbing

A heurística tenta alterar blocos das aulas, aceitando movimentos que melhoram a pontuação e, ocasionalmente, piores, conforme uma temperatura decrescente. Movimentos que violem hard constraints são rejeitados.


In [9]:
def hill_climbing(solucao_inicial, iteracoes=10000, temp_inicial=10.0, decaimento=0.995):
    melhor_solucao = copy.deepcopy(solucao_inicial)
    melhor_pontuacao = pontuacao(melhor_solucao)
    variaveis = [v for v in melhor_solucao.keys() if v.startswith("aula_")]

    temperatura = temp_inicial

    for i in range(iteracoes):
        var = random.choice(variaveis)
        aula_atual = melhor_solucao[var]
        blocos_possiveis = [b for b in range(1, 21) if b != aula_atual.bloco]
        bloco_novo = random.choice(blocos_possiveis)

        if violates_hard_constraints_for_move(melhor_solucao, var, bloco_novo, variaveis):
            continue

        nova_solucao = copy.deepcopy(melhor_solucao)
        nova_solucao[var].bloco = bloco_novo

        nova_pontuacao = pontuacao(nova_solucao)
        delta = nova_pontuacao - melhor_pontuacao

        if delta < 0 or random.random() < math.exp(-delta / temperatura):
            melhor_solucao = nova_solucao
            melhor_pontuacao = nova_pontuacao

        temperatura *= decaimento

    return melhor_solucao
